# Experiment: Identify 250 hPa divergence causes along the backward trajectory

Objective:
- Visualize 250 hPa divergence around the 72 h backward trajectory ending over Vancouver at 2021-11-12 15:00 UTC.
- Use discrete divergence bins so convergence and divergence zones are easier to compare frame-to-frame.
- Sample upper-level divergence at 6-hour trajectory points to see when the parcel enters or exits strongly divergent flow.


In [1]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    CARTOPY_AVAILABLE = True
except ModuleNotFoundError:
    CARTOPY_AVAILABLE = False

plt.rcParams["figure.dpi"] = 130
{"cartopy_available": CARTOPY_AVAILABLE}


{'cartopy_available': True}

## Plan

- Hypothesis: if the Vancouver-targeted parcel is forced by an upper-level divergent jet region, the 250 hPa field should show coherent positive divergence near or upstream of key trajectory segments.
- Variables to sweep: map time, frame step, divergence bin width, and map extent.
- Metrics to record: parcel-sampled divergence, spatial context relative to the full trajectory, and whether each 6-hour point sits in divergence or convergence.


In [2]:
DATA_FILE = "era5_2021-nov_250-500-925_uv_pv_gph.nc"
DIVERGENCE_FILE = "era5_2021-nov_250-500-925_divergence_vertical_velocity.nc"


def resolve_data_path(filename: str) -> Path:
    candidates = [
        Path("../data") / filename,
        Path("data") / filename,
        Path.cwd() / "data" / filename,
        Path.cwd().parent / "data" / filename,
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Checked: {checked}")


UV_PATH = resolve_data_path(DATA_FILE)
DIVERGENCE_PATH = resolve_data_path(DIVERGENCE_FILE)
MAP_EXTENT = {"lon_min": 120.0, "lon_max": 265.0, "lat_min": 30.0, "lat_max": 72.0}
TRAJECTORY_START = {
    "lat": 49.28,
    "lon": -123.12,
    "time": "2021-11-12T15:00:00",
    "pressure_level": 925,
}
HOURS_BACK = 72
SUBSTEPS = 4
FRAME_STEP_HOURS = 6
TARGET_DIVERGENCE_LEVEL_HPA = 250.0
DIVERGENCE_BIN_WIDTH_S1 = 1.0e-5
GPH_CONTOUR_STEP_M = 40.0

uv_ds = xr.open_dataset(UV_PATH)
div_ds = xr.open_dataset(DIVERGENCE_PATH)
time_coord = "valid_time" if "valid_time" in div_ds.coords else "time"
if time_coord not in div_ds.coords:
    raise RuntimeError("Expected a valid_time/time coordinate in the divergence dataset.")

lat_descending = float(div_ds["latitude"].values[0]) > float(div_ds["latitude"].values[-1])
lat_slice = (
    slice(MAP_EXTENT["lat_max"], MAP_EXTENT["lat_min"])
    if lat_descending
    else slice(MAP_EXTENT["lat_min"], MAP_EXTENT["lat_max"])
)

gph_925 = xr.apply_ufunc(
    np.divide,
    uv_ds["z"].sel(pressure_level=925, method="nearest"),
    9.80665,
)
gph_925.name = "gph_925_m"

{
    "uv_path": str(UV_PATH),
    "divergence_path": str(DIVERGENCE_PATH),
    "uv_dims": dict(uv_ds.sizes),
    "divergence_dims": dict(div_ds.sizes),
}


/home/dmmsp/anaconda3/envs/water-transport-in-atmosphere/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'uv_path': '/home/dmmsp/Projects/water-transport-in-atmosphere/data/era5_2021-nov_250-500-925_uv_pv_gph.nc',
 'divergence_path': '/home/dmmsp/Projects/water-transport-in-atmosphere/data/era5_2021-nov_250-500-925_divergence_vertical_velocity.nc',
 'uv_dims': {'valid_time': 720,
  'pressure_level': 3,
  'latitude': 721,
  'longitude': 1440},
 'divergence_dims': {'valid_time': 720,
  'pressure_level': 3,
  'latitude': 721,
  'longitude': 1440}}

In [ ]:
def backward_integrate_trajectory_uv(
    ds: xr.Dataset,
    start_lat: float,
    start_lon: float,
    start_time: str | pd.Timestamp,
    pressure_level: int | float = 925,
    hours_back: int = 72,
    substeps: int = 4,
    earth_radius_m: float = 6_371_000.0,
) -> pd.DataFrame:
    """Backward trajectory using ERA5 u/v winds."""
    if substeps < 1:
        raise ValueError("substeps must be >= 1")

    ds_uv = ds[["u", "v"]].sel(pressure_level=pressure_level)
    t0 = pd.Timestamp(start_time).to_datetime64()
    t_nearest = ds_uv["valid_time"].sel(valid_time=t0, method="nearest").values

    window_start = np.datetime64(t_nearest) - np.timedelta64(hours_back + 2, "h")
    ds_uv = ds_uv.sel(valid_time=slice(window_start, np.datetime64(t_nearest))).load()

    lat = float(start_lat)
    lon = float(start_lon) % 360.0
    t_curr = np.datetime64(t_nearest)
    t_min = np.datetime64(pd.to_datetime(ds_uv["valid_time"].values).min().to_datetime64())

    records = [
        {
            "step_hour": 0,
            "valid_time": pd.Timestamp(t_nearest),
            "latitude": lat,
            "longitude": lon,
        }
    ]

    dt_hour_s = 3600.0
    dt_sub_s = dt_hour_s / substeps

    for hour in range(1, hours_back + 1):
        if t_curr - np.timedelta64(1, "h") < t_min:
            break

        lat_step = lat
        lon_step = lon

        for substep in range(substeps):
            sec_back = (substep + 0.5) * dt_sub_s
            t_mid = t_curr - np.timedelta64(int(sec_back), "s")

            u_ms = float(
                ds_uv["u"].interp(
                    valid_time=[t_mid],
                    latitude=[lat_step],
                    longitude=[lon_step],
                    kwargs={"fill_value": "extrapolate"},
                ).squeeze().values
            )
            v_ms = float(
                ds_uv["v"].interp(
                    valid_time=[t_mid],
                    latitude=[lat_step],
                    longitude=[lon_step],
                    kwargs={"fill_value": "extrapolate"},
                ).squeeze().values
            )

            dlat_deg = np.degrees((v_ms * dt_sub_s) / earth_radius_m)
            coslat = max(np.cos(np.radians(lat_step)), 1e-6)
            dlon_deg = np.degrees((u_ms * dt_sub_s) / (earth_radius_m * coslat))

            lat_step = float(np.clip(lat_step - dlat_deg, -89.75, 89.75))
            lon_step = float((lon_step - dlon_deg) % 360.0)

        t_curr = t_curr - np.timedelta64(1, "h")
        lat = lat_step
        lon = lon_step

        records.append(
            {
                "step_hour": hour,
                "valid_time": pd.Timestamp(t_curr),
                "latitude": lat,
                "longitude": lon,
            }
        )

    return pd.DataFrame(records)


trajectory_df = backward_integrate_trajectory_uv(
    uv_ds,
    start_lat=TRAJECTORY_START["lat"],
    start_lon=TRAJECTORY_START["lon"],
    start_time=TRAJECTORY_START["time"],
    pressure_level=TRAJECTORY_START["pressure_level"],
    hours_back=HOURS_BACK,
    substeps=SUBSTEPS,
)
trajectory_df["longitude_360"] = trajectory_df["longitude"] % 360.0
trajectory_df.head()


In [ ]:
def frame_rows_from_trajectory(trajectory: pd.DataFrame, frame_step_hours: int = 6) -> pd.DataFrame:
    frame_rows = (
        trajectory.loc[trajectory["step_hour"] % frame_step_hours == 0]
        .sort_values("step_hour")
        .reset_index(drop=True)
    )
    if frame_rows.empty:
        raise RuntimeError("No trajectory rows matched the requested frame spacing.")
    return frame_rows


def symmetric_discrete_levels(data_array: xr.DataArray, bin_width_s1: float = 1.0e-5):
    max_abs = float(np.nanmax(np.abs(data_array.values)))
    if not np.isfinite(max_abs):
        raise RuntimeError("Non-finite divergence range encountered.")
    if max_abs < bin_width_s1:
        max_abs = bin_width_s1
    vmax = float(np.ceil(max_abs / bin_width_s1) * bin_width_s1)
    levels = np.arange(-vmax, vmax + bin_width_s1, bin_width_s1)
    n_bins = max(len(levels) - 1, 1)
    cmap = plt.get_cmap("RdBu_r", n_bins)
    norm = mcolors.BoundaryNorm(levels, ncolors=n_bins, clip=True)
    return levels, cmap, norm


trajectory_line = trajectory_df.sort_values("step_hour", ascending=False).copy()
frame_rows = frame_rows_from_trajectory(trajectory_df, FRAME_STEP_HOURS)
frame_times = xr.DataArray(
    pd.to_datetime(frame_rows["valid_time"]).to_numpy(dtype="datetime64[ns]"),
    dims="frame",
)

div_region = div_ds["d"].sel(pressure_level=TARGET_DIVERGENCE_LEVEL_HPA, method="nearest").sel(
    longitude=slice(MAP_EXTENT["lon_min"], MAP_EXTENT["lon_max"]),
    latitude=lat_slice,
)
gph_region = gph_925.sel(
    longitude=slice(MAP_EXTENT["lon_min"], MAP_EXTENT["lon_max"]),
    latitude=lat_slice,
)

div_frames = div_region.sel({time_coord: frame_times}, method="nearest").load()
gph_frames = gph_region.sel(valid_time=frame_times, method="nearest").load()
discrete_levels, divergence_cmap, divergence_norm = symmetric_discrete_levels(
    div_frames,
    DIVERGENCE_BIN_WIDTH_S1,
)


def plot_divergence_with_trajectory(frame_index: int = 0):
    row = frame_rows.iloc[frame_index]
    divergence_frame = div_frames.isel(frame=frame_index)
    gph_frame = gph_frames.isel(frame=frame_index)
    lon_curr = float(row["longitude_360"])
    lat_curr = float(row["latitude"])
    frame_time = pd.Timestamp(row["valid_time"]).round("h")
    step_hour = int(row["step_hour"])

    gph_min = float(gph_frame.min().values)
    gph_max = float(gph_frame.max().values)
    gph_start = math.floor(gph_min / GPH_CONTOUR_STEP_M) * GPH_CONTOUR_STEP_M
    gph_stop = math.ceil(gph_max / GPH_CONTOUR_STEP_M) * GPH_CONTOUR_STEP_M + GPH_CONTOUR_STEP_M
    gph_levels = np.arange(gph_start, gph_stop, GPH_CONTOUR_STEP_M)

    if CARTOPY_AVAILABLE:
        fig = plt.figure(figsize=(13, 6.5))
        ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
        ax.set_extent(
            [
                MAP_EXTENT["lon_min"],
                MAP_EXTENT["lon_max"],
                MAP_EXTENT["lat_min"],
                MAP_EXTENT["lat_max"],
            ],
            crs=ccrs.PlateCarree(),
        )
        ax.coastlines(resolution="110m", linewidth=0.8, color="black")
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="black")
        mesh = ax.pcolormesh(
            divergence_frame["longitude"],
            divergence_frame["latitude"],
            divergence_frame,
            cmap=divergence_cmap,
            norm=divergence_norm,
            shading="auto",
            transform=ccrs.PlateCarree(),
            zorder=1,
        )
        contours = ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=gph_levels,
            colors="black",
            linewidths=0.7,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=5,
        )
        ax.clabel(contours, inline=True, fontsize=7, fmt="%d")
        ax.plot(
            trajectory_line["longitude_360"],
            trajectory_line["latitude"],
            color="white",
            linewidth=2.0,
            alpha=0.95,
            transform=ccrs.PlateCarree(),
            zorder=6,
        )
        ax.scatter(
            trajectory_line["longitude_360"],
            trajectory_line["latitude"],
            c="black",
            s=15,
            alpha=0.35,
            transform=ccrs.PlateCarree(),
            zorder=7,
        )
        ax.scatter(
            [lon_curr],
            [lat_curr],
            marker="x",
            s=120,
            c="deepskyblue",
            linewidths=2.2,
            transform=ccrs.PlateCarree(),
            zorder=8,
        )
        gridlines = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5, linestyle="--")
        gridlines.top_labels = False
        gridlines.right_labels = False
    else:
        fig, ax = plt.subplots(figsize=(13, 6.5))
        mesh = ax.pcolormesh(
            divergence_frame["longitude"],
            divergence_frame["latitude"],
            divergence_frame,
            cmap=divergence_cmap,
            norm=divergence_norm,
            shading="auto",
        )
        contours = ax.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=gph_levels,
            colors="black",
            linewidths=0.7,
            alpha=0.7,
        )
        ax.clabel(contours, inline=True, fontsize=7, fmt="%d")
        ax.plot(
            trajectory_line["longitude_360"],
            trajectory_line["latitude"],
            color="black",
            linewidth=2.0,
            alpha=0.95,
        )
        ax.scatter(
            trajectory_line["longitude_360"],
            trajectory_line["latitude"],
            c="white",
            edgecolors="black",
            s=15,
            alpha=0.65,
        )
        ax.scatter([lon_curr], [lat_curr], marker="x", s=120, c="deepskyblue", linewidths=2.2)
        ax.set_xlim(MAP_EXTENT["lon_min"], MAP_EXTENT["lon_max"])
        ax.set_ylim(MAP_EXTENT["lat_min"], MAP_EXTENT["lat_max"])
        ax.set_xlabel("Longitude (degrees east)")
        ax.set_ylabel("Latitude")
        ax.grid(alpha=0.3, linestyle="--")

    ax.set_title(
        "250 hPa discrete divergence with backward trajectory | "
        f"{frame_time:%Y-%m-%d %H:%M UTC} (t-{step_hour}h)"
    )
    cbar = plt.colorbar(
        mesh,
        ax=ax,
        boundaries=discrete_levels,
        orientation="horizontal",
        pad=0.06,
        shrink=0.9,
    )
    cbar.set_label("Divergence (s$^{-1}$): negative=convergence, positive=divergence")
    plt.tight_layout()
    return fig, ax


fig, ax = plot_divergence_with_trajectory(frame_index=0)
plt.show()


In [ ]:
def plot_divergence_panel_overview(max_panels: int = 6):
    panel_rows = frame_rows.iloc[:max_panels].copy()
    n_panels = len(panel_rows)
    ncols = 2
    nrows = math.ceil(n_panels / ncols)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(16, 4.7 * nrows),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()
    mesh = None

    for axis, (_, row) in zip(axes, panel_rows.iterrows()):
        frame_index = int(row.name)
        divergence_frame = div_frames.isel(frame=frame_index)
        gph_frame = gph_frames.isel(frame=frame_index)
        mesh = axis.pcolormesh(
            divergence_frame["longitude"],
            divergence_frame["latitude"],
            divergence_frame,
            cmap=divergence_cmap,
            norm=divergence_norm,
            shading="auto",
        )
        axis.contour(
            gph_frame["longitude"],
            gph_frame["latitude"],
            gph_frame,
            levels=np.arange(460.0, 980.0, GPH_CONTOUR_STEP_M),
            colors="black",
            linewidths=0.45,
            alpha=0.45,
        )
        axis.plot(
            trajectory_line["longitude_360"],
            trajectory_line["latitude"],
            color="black",
            linewidth=1.3,
        )
        axis.scatter(
            trajectory_line["longitude_360"],
            trajectory_line["latitude"],
            c="white",
            edgecolors="black",
            s=10,
            alpha=0.5,
        )
        axis.scatter(
            [float(row["longitude_360"])],
            [float(row["latitude"])],
            marker="x",
            s=70,
            c="deepskyblue",
            linewidths=1.8,
        )
        axis.set_xlim(MAP_EXTENT["lon_min"], MAP_EXTENT["lon_max"])
        axis.set_ylim(MAP_EXTENT["lat_min"], MAP_EXTENT["lat_max"])
        axis.grid(alpha=0.25, linestyle="--")
        axis.set_title(
            f"{pd.Timestamp(row['valid_time']):%m-%d %H:%M UTC} | t-{int(row['step_hour'])}h"
        )

    for axis in axes[n_panels:]:
        axis.set_visible(False)

    if mesh is not None:
        cbar = fig.colorbar(
            mesh,
            ax=[axis for axis in axes[:n_panels]],
            boundaries=discrete_levels,
            orientation="horizontal",
            pad=0.04,
            shrink=0.92,
        )
        cbar.set_label("Divergence (s$^{-1}$): negative=convergence, positive=divergence")

    for axis in axes[-ncols:]:
        if axis.get_visible():
            axis.set_xlabel("Longitude (degrees east)")
    for axis in axes[::ncols]:
        if axis.get_visible():
            axis.set_ylabel("Latitude")

    return fig, axes


fig, axes = plot_divergence_panel_overview(max_panels=6)
plt.show()


## Results

- The map view shows where the trajectory sits relative to upper-level convergence and divergence at each 6-hour step.
- The table and bar chart below make it easier to identify which part of the path is most strongly divergent.
- Positive values imply divergent flow aloft; negative values imply convergent flow aloft.


In [ ]:
sampled_trajectory = frame_rows.copy()
sampled_trajectory["divergence_250_s-1"] = [
    float(
        div_region.sel(
            {
                time_coord: np.datetime64(timestamp),
                "latitude": latitude,
                "longitude": longitude,
            },
            method="nearest",
        ).values
    )
    for timestamp, latitude, longitude in zip(
        sampled_trajectory["valid_time"],
        sampled_trajectory["latitude"],
        sampled_trajectory["longitude_360"],
    )
]
sampled_trajectory["classification"] = np.where(
    sampled_trajectory["divergence_250_s-1"] >= 0.0,
    "divergence",
    "convergence",
)
sampled_trajectory[
    [
        "step_hour",
        "valid_time",
        "latitude",
        "longitude",
        "divergence_250_s-1",
        "classification",
    ]
]


In [ ]:
plot_table = sampled_trajectory.sort_values("step_hour").copy()
bar_colors = np.where(plot_table["divergence_250_s-1"] >= 0.0, "firebrick", "navy")

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axhline(0.0, color="black", linewidth=1.0)
ax.bar(
    plot_table["step_hour"],
    plot_table["divergence_250_s-1"],
    width=4.5,
    color=bar_colors,
    alpha=0.8,
)
ax.plot(
    plot_table["step_hour"],
    plot_table["divergence_250_s-1"],
    color="black",
    linewidth=1.1,
    marker="o",
    markersize=3,
)
ax.set_xlabel("Backward trajectory step hour")
ax.set_ylabel("250 hPa divergence (s$^{-1}$)")
ax.set_title("250 hPa divergence sampled at 6-hour backward-trajectory points")
ax.grid(alpha=0.25, linestyle="--")
ax.invert_xaxis()
plt.tight_layout()
plt.show()

sampled_trajectory.sort_values("divergence_250_s-1", ascending=False).head(5)


## Next steps

- Tune `DIVERGENCE_BIN_WIDTH_S1` if you want fewer or more aggressive discrete color bins.
- Change `TRAJECTORY_START["time"]` to compare other arrival times over Vancouver.
- Add 250 hPa wind speed or vertical velocity next if you want to separate jet-related divergence from ascent/descent signatures.
